## SECTION 1: Google Colab Pipeline (Data Preprocessing, Engineering, Training, & Export)

This section covers the data engineering, model training, and saving of the Random Forest model for your demand forecasting project.

**Step 1: Import Libraries and Load `salesdaily.csv`**

In [8]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


### 1. Load `salesdaily.csv` and Parse Date Column

We will load the `salesdaily.csv` file into a pandas DataFrame and parse the 'Datum' column to datetime.

In [12]:
# Load the provided 'salesdaily.csv' data
try:
    df = pd.read_csv('salesdaily.csv')
    print("salesdaily.csv loaded successfully.")
except FileNotFoundError:
    print("Error: salesdaily.csv not found. Please ensure it's uploaded to your Colab environment.")
    # Fallback to simulation if file not found, for demonstration purposes
    dates = pd.date_range(start='2020-01-01', end='2022-12-31', freq='D')
    np.random.seed(42)
    data = {
        'datum': dates,
        'M01AB': np.random.randint(100, 500, size=len(dates)) + np.sin(np.arange(len(dates)) / 30) * 100 + np.random.normal(0, 20, size=len(dates)),
        'M01AE': np.random.randint(50, 300, size=len(dates)) + np.cos(np.arange(len(dates)) / 60) * 80 + np.random.normal(0, 15, size=len(dates)),
        'R03': np.random.randint(200, 800, size=len(dates)) + np.sin(np.arange(len(dates)) / 90) * 150 + np.random.normal(0, 30, size=len(dates))
    }
    df = pd.DataFrame(data)
    print("Falling back to mock data as salesdaily.csv was not found.")

# Ensure 'datum' column exists and convert to datetime
if 'datum' in df.columns:
    df['datum'] = pd.to_datetime(df['datum'])
    df = df.set_index('datum').sort_index()
    print("'datum' column parsed to datetime and set as index.")
else:
    print("Error: 'datum' column not found in the dataset. Please check the CSV file structure.")

display(df.head())

salesdaily.csv loaded successfully.
'datum' column parsed to datetime and set as index.


,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
datum,,,,,,,,,,,,
2014-01-02,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0,2014,1,248,Thursday
2014-01-03,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0,2014,1,276,Friday
2014-01-04,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0,2014,1,276,Saturday
2014-01-05,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0,2014,1,276,Sunday
2014-01-06,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0,2014,1,276,Monday


### 2. Feature Engineering

We will extract 'Year', 'Month', 'DayOfWeek' from the `datum` index and create 7-day rolling demand lag and rolling mean features for the specified ATC codes (`M01AB`, `M01AE`, `R03`).

In [34]:
# Create time-based features from the index
df['Year'] = df.index.year
df['Month'] = df.index.month
df['DayOfWeek'] = df.index.dayofweek # Monday=0, Sunday=6

# Dynamically identify target ATC columns
# Updated regex to be more flexible, capturing codes like 'R03', 'N05B', 'N05C' as well as 'M01AB', etc.
import re
atc_columns = [col for col in df.columns if re.match(r'^(M|N|R)\d{2}([A-Z]{1,2})?$', col)]
# Ensure the order is consistent for feature generation later
atc_columns.sort()
print(f"Dynamically identified ATC columns: {atc_columns}")

# Let's check which of the requested ATC columns are actually in the df
actual_atc_columns = [col for col in atc_columns if col in df.columns]
if len(actual_atc_columns) < len(atc_columns):
    print(f"Warning: Not all specified ATC columns found in salesdaily.csv. Using: {actual_atc_columns}")

# Implement Fix 1: Change Target Variable to a Moving Average
for col in actual_atc_columns:
    df[f'{col}_smoothed'] = df[col].rolling(window=7, min_periods=1).mean()

# Implement Fix 2: Create Stronger "Lag" Features and Rolling Mean
for col in actual_atc_columns:
    df[f'{col}_Lag1'] = df[col].shift(1)
    df[f'{col}_Lag2'] = df[col].shift(2)
    df[f'{col}_Lag7'] = df[col].shift(7)
    df[f'{col}_RollingMean7'] = df[col].rolling(window=7).mean().shift(1)

# Drop rows with NaN values resulting from lag/rolling features and smoothing
df.dropna(inplace=True)

print("Feature engineering complete with smoothed targets and additional lag features.")
display(df.head())

Dynamically identified ATC columns: ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']
Feature engineering complete with smoothed targets and additional lag features.


,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,...,N05B_RollingMean7,N05C_Lag1,N05C_Lag2,N05C_Lag7,N05C_RollingMean7,R06_smoothed,R06_Lag1,R06_Lag2,R06_Lag7,R06_RollingMean7
datum,,,,,,,,,,,,,,,,,,,,,
2014-01-30,3.02,1.34,2.4,25.5,7.0,1.0,3.0,2.0,2014,1,...,12.285714,1.0,5.0,3.0,1.714286,1.857143,2.0,5.0,3.0,2.000000
2014-01-31,1.00,2.68,7.1,26.9,9.0,0.0,1.0,0.0,2014,1,...,13.000000,1.0,1.0,1.0,1.428571,1.857143,2.0,2.0,0.0,1.857143
2014-02-01,4.33,4.32,5.0,43.0,13.0,1.0,14.0,0.0,2014,2,...,12.428571,0.0,1.0,0.0,1.285714,1.714286,0.0,2.0,1.0,1.857143
2014-02-02,7.00,3.00,0.2,13.5,6.0,2.0,8.0,0.0,2014,2,...,11.714286,1.0,0.0,0.0,1.428571,1.714286,0.0,0.0,0.0,1.714286
2014-02-03,5.00,1.00,8.5,32.4,16.0,1.0,1.0,0.0,2014,2,...,12.000000,2.0,1.0,2.0,1.714286,1.285714,0.0,0.0,3.0,1.714286


### 3. Splitting the Data Sequentially (Time-Series Split)

We will split the data into training and testing sets while preserving the chronological order, which is crucial for time-series forecasting. We'll use 80% for training and 20% for testing.

In [35]:
# Define features (X) and target (y) for each ATC code
# We will use the actual_atc_columns discovered during feature engineering.

# Update y_cols to use the smoothed target variables
y_cols = [f'{col}_smoothed' for col in actual_atc_columns]

# Update X_cols to include the new lag features
X_cols = ['Year', 'Month', 'DayOfWeek'] + \
         [f'{col}_Lag1' for col in actual_atc_columns] + \
         [f'{col}_Lag2' for col in actual_atc_columns] + \
         [f'{col}_Lag7' for col in actual_atc_columns] + \
         [f'{col}_RollingMean7' for col in actual_atc_columns]

# Ensure all feature columns exist after dropping NaNs
X_cols = [col for col in X_cols if col in df.columns]

X = df[X_cols]
y = df[y_cols] # Use the smoothed targets

# Time-series split: 80% train, 20% test
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Data split into training (n={len(X_train)}) and testing (n={len(X_test)}) sets.")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

Data split into training (n=1662) and testing (n=416) sets.
X_train shape: (1662, 35), y_train shape: (1662, 8)
X_test shape: (416, 35), y_test shape: (416, 8)


### 4. Train Baseline Linear Regression Model

We'll train a Linear Regression model as a baseline and evaluate its performance using $R^2$ and RMSE.

In [36]:
from sklearn.preprocessing import StandardScaler # Added import

# Train Linear Regression model
print("\n--- Training Linear Regression Model (with scaling and new features) ---")

# Implement Fix 3: Scale/Standardize for Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train) # Train on scaled X, smoothed y

# Make predictions
y_pred_lr = linear_model.predict(X_test_scaled) # Predict on scaled X_test

# Evaluate Linear Regression model
r2_lr = r2_score(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print(f"Linear Regression R^2 (with smoothed target and new features): {r2_lr:.4f}")
print(f"Linear Regression RMSE (with smoothed target and new features): {rmse_lr:.4f}")


--- Training Linear Regression Model (with scaling and new features) ---
Linear Regression R^2 (with smoothed target and new features): 0.8970
Linear Regression RMSE (with smoothed target and new features): 0.8245
